# 3 · The Jacobian

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rudip1/learn-manipulator-control/blob/main/2_notebooks/exercises/03_jacobian.ipynb)

Theory: [`1_theory/03_jacobian.md`](../../1_theory/03_jacobian.md) · notation:
[`00_notation.md`](../../1_theory/00_notation.md)

In [ ]:
# Install the module when it is not importable yet (Colab, fresh environment).
import importlib.util
if importlib.util.find_spec("manipulator_control") is None:
    %pip install -q git+https://github.com/Rudip1/learn-manipulator-control
if importlib.util.find_spec("pinocchio") is None:
    %pip install -q pin

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import manipulator_control as mc
from manipulator_control import plotting
from manipulator_control.pinocchio_ref import frame_jacobian, model_from_chain

plt.rcParams.update({"figure.dpi": 80, "axes.grid": True})
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(3)

## Recap

- The geometric Jacobian maps joint velocities to end-effector velocity, $(\dot p_e, \omega_e) = J(q)\dot q$
  (eq. 3.1), rows linear first.
- Column $i$: $(z_{i-1}\times(p_e - o_{i-1}),\ z_{i-1})$ for a revolute joint, $(z_{i-1}, 0)$ for a prismatic one
  (eq. 3.2). The Jacobian of frame $k$ uses $o_k$ and only joints $1..k$ (eq. 3.4).
- From the product of exponentials: space Jacobian $J_s$ (eq. 3.5), body Jacobian $J_b = \mathrm{Ad}_{T^{-1}}J_s$
  (eq. 3.6), and $J = \begin{bmatrix} I & -[p_e]_\times \\ 0 & I\end{bmatrix}J_s$ (eq. 3.7).
- The analytic Jacobian differentiates a minimal pose $x = (p, \phi, \vartheta, \psi)$:
  $J_A = \mathrm{blockdiag}(I, T^{-1})J$, undefined where $\cos\vartheta = 0$ (eqs. 3.8–3.9).
- Central differences check any Jacobian; the orientation column uses $\log(R_+R_-^T)/2h$ (eq. 3.10).

## What a column means

Drive one joint of a planar arm at unit speed and integrate: the end-effector starts out along column $J_{P,i}$.

In [ ]:
arm = mc.robots.planar([0.75, 0.5, 0.5])
q0 = np.array([0.3, 0.9, 0.6])
J = mc.geometric_jacobian(arm, q0)
print("J (rows vx vy vz wx wy wz) =\n", J)

fig, ax = plt.subplots(figsize=(5, 5))
plotting.draw_planar_arm(ax, arm.frames(q0), color="0.3")
p0 = arm.end_effector(q0)[:2, 3]
for i in range(3):
    path = []
    for t in np.linspace(0, 0.4, 20):
        q = q0.copy(); q[i] += t
        path.append(arm.end_effector(q)[:2, 3])
    path = np.array(path)
    ax.plot(path[:, 0], path[:, 1], color=f"C{i}", label=f"joint {i + 1} moving")
    ax.annotate("", xy=p0 + 0.4 * J[:2, i], xytext=p0, arrowprops=dict(arrowstyle="->", color=f"C{i}"))
plotting.planar_axes(ax, title="end-effector paths and Jacobian columns")
ax.set_xlim(-0.6, 1.4); ax.set_ylim(0, 2.0); ax.legend(fontsize=8)
plt.show()

## Three independent checks on the PUMA 560

The DH construction (eq. 3.2), finite differences (eq. 3.10), the product-of-exponentials route (eq. 3.7) and
Pinocchio should all give the same matrix.

In [ ]:
puma = mc.robots.puma560()
poe = mc.to_poe(puma)
model, data, ids = model_from_chain(puma)
worst = {"finite differences": 0.0, "POE, eq. (3.7)": 0.0, "Pinocchio": 0.0}
for _ in range(200):
    q = rng.uniform(-np.pi, np.pi, 6)
    J = mc.geometric_jacobian(puma, q)
    p = puma.end_effector(q)[:3, 3]
    shift = np.eye(6); shift[:3, 3:] = -mc.skew(p)
    worst["finite differences"] = max(worst["finite differences"], np.abs(J - mc.numerical_jacobian(puma, q)).max())
    worst["POE, eq. (3.7)"] = max(worst["POE, eq. (3.7)"], np.abs(J - shift @ mc.space_jacobian(poe, q)).max())
    worst["Pinocchio"] = max(worst["Pinocchio"], np.abs(J - frame_jacobian(model, data, ids[-1], q)).max())
for name, err in worst.items():
    print(f"max |J - {name}| over 200 configurations: {err:.1e}")

## Geometric versus analytic

The position rows agree; the orientation rows do not. $\det T = \cos\vartheta$, so the analytic Jacobian
degrades as the pitch of the end-effector approaches $\pm 90°$ while the geometric one does not care.

In [ ]:
q = np.array([0.3, 0.8, -0.5, 0.2, 0.9, -0.4])
JA, JG = mc.analytic_jacobian(puma, q), mc.geometric_jacobian(puma, q)
print("position rows equal:", np.allclose(JA[:3], JG[:3]))
print("orientation rows, geometric:\n", JG[3:], "\norientation rows, analytic:\n", JA[3:])
print("analytic vs finite differences of rpy:", np.abs(JA - mc.numerical_analytic_jacobian(puma, q)).max())

## ✏️ Exercises

### ✏️ 1. The planar two-link Jacobian by hand

Differentiate eq. (2.3) and write the $2\times 2$ position Jacobian of the planar two-link arm.

In [ ]:
l1, l2 = 0.75, 0.5
def jacobian_2r(q):
    # ✏️ your code here
    raise NotImplementedError("exercise")

In [ ]:
two = mc.robots.planar([l1, l2])
for _ in range(20):
    q = rng.uniform(-np.pi, np.pi, 2)
    assert np.allclose(jacobian_2r(q), mc.geometric_jacobian(two, q)[:2])
print("✓ jacobian_2r")

### ✏️ 2. Build the Jacobian from the frames

Implement eq. (3.2)–(3.3) for an all-revolute arm, using only `chain.frames(q)` (remember the list is
$T_0, \dots, T_n, T_e$ and joint $i$ uses frame $i-1$).

In [ ]:
def my_jacobian(chain, q):
    # ✏️ your code here
    raise NotImplementedError("exercise")

In [ ]:
for _ in range(10):
    q = rng.uniform(-np.pi, np.pi, 6)
    assert np.allclose(my_jacobian(puma, q), mc.geometric_jacobian(puma, q))
print("✓ my_jacobian")

### ✏️ 3. Finite differences for the position rows

Implement the central difference of eq. (3.10) for $J_P$ only.

In [ ]:
def numerical_position_jacobian(chain, q, h=1e-6):
    # ✏️ your code here
    raise NotImplementedError("exercise")

In [ ]:
q = rng.uniform(-np.pi, np.pi, 6)
assert np.allclose(numerical_position_jacobian(puma, q), mc.geometric_jacobian(puma, q)[:3], atol=1e-8)
print("✓ numerical_position_jacobian")

### ✏️ 4. The roll–pitch–yaw rate matrix

Derive $T(\phi, \vartheta, \psi)$ of eq. (3.8): angular velocity is the sum of the three rates, each about its
own axis — yaw about $z_0$, pitch about $R_z(\psi)y_0$, roll about $R_z(\psi)R_y(\vartheta)x_0$. Implement it
with `mc.rot_z` / `mc.rot_y`, then form the analytic Jacobian (eq. 3.9).

In [ ]:
def rpy_rates(rpy):
    # ✏️ your code here
    raise NotImplementedError("exercise")

def my_analytic_jacobian(chain, q):
    # ✏️ your code here
    raise NotImplementedError("exercise")

In [ ]:
rpy = np.array([0.2, -0.6, 1.3])
assert np.allclose(rpy_rates(rpy), mc.rpy_rate_matrix(rpy))
assert np.allclose(my_analytic_jacobian(puma, q), mc.analytic_jacobian(puma, q))
print("✓ rpy_rates and my_analytic_jacobian")

## 🔨 Break it

### 🔨 1. The finite-difference step

Truncation error grows like $h^2$, rounding error like $\varepsilon/h$ (section 3.5). Sweep $h$ and see the V.

In [ ]:
q = np.array([0.3, 0.8, -0.5, 0.2, 0.9, -0.4])
J = mc.geometric_jacobian(puma, q)
hs = np.logspace(-12, -1, 45)
err_central = [np.abs(mc.numerical_jacobian(puma, q, h) - J).max() for h in hs]

def forward_difference(chain, q, h):
    T0 = chain.end_effector(q)
    J = np.zeros((6, chain.dof))
    for i in range(chain.dof):
        dq = np.zeros(chain.dof); dq[i] = h
        T1 = chain.end_effector(q + dq)
        J[:3, i] = (T1[:3, 3] - T0[:3, 3]) / h
        J[3:, i] = mc.log_so3(T1[:3, :3] @ T0[:3, :3].T) / h
    return J

err_forward = [np.abs(forward_difference(puma, q, h) - J).max() for h in hs]
plt.figure(figsize=(5.5, 3.5))
plt.loglog(hs, err_central, "o-", ms=3, label="central")
plt.loglog(hs, err_forward, "s-", ms=3, label="forward")
plt.xlabel("step h"); plt.ylabel("max |error|"); plt.legend(); plt.title("finite-difference Jacobian error")
plt.show()
print(f"best central step: h = {hs[int(np.argmin(err_central))]:.0e}, error {min(err_central):.1e}")

### 🔨 2. A link Jacobian with the wrong point

The Jacobian of frame $k$ needs $o_k$ in the cross product. Using the end-effector origin instead (a mistake that
is easy to make when adapting end-effector code) gives a matrix that looks reasonable and is wrong. Compare
both with finite differences of the elbow position (frame 3 of the PUMA).

In [ ]:
def link_jacobian_wrong_point(chain, q, k):
    frames = chain.frames(q)
    p = frames[-1][:3, 3]  # BUG: should be frames[k][:3, 3]
    J = np.zeros((6, chain.dof))
    for i in range(k):
        z, o = frames[i][:3, 2], frames[i][:3, 3]
        J[:3, i] = np.cross(z, p - o); J[3:, i] = z
    return J

k = 3
J_fd = mc.numerical_jacobian(puma, q, 1e-6, k)
print("correct link Jacobian, error vs finite differences:", np.abs(mc.link_jacobian(puma, q, k) - J_fd).max())
print("end-effector point used, error vs finite differences:", np.abs(link_jacobian_wrong_point(puma, q, k) - J_fd).max())

### 🔨 3. The analytic Jacobian near gimbal lock

Move joint 5 so the end-effector pitch sweeps up to $90°$ (the extracted pitch then folds back, since it lives
in $[-90°, 90°]$). The arm is nowhere near singular — the geometric
Jacobian's condition number stays modest — but the analytic one explodes.

In [ ]:
base_q = np.array([0.0, 0.5, -0.3, 0.0, 0.0, 0.0])
q5s = np.linspace(-2.6, -0.9, 400)  # the hand's pitch is -(q2 + q3 + q5): it reaches 90° at q5 = -pi/2 - 0.2
pitch, cond_G, cond_A = [], [], []
for q5 in q5s:
    qq = base_q.copy(); qq[4] = q5
    T = puma.end_effector(qq)
    pitch.append(mc.rpy_from_rotation(T[:3, :3])[1])
    cond_G.append(np.linalg.cond(mc.geometric_jacobian(puma, qq)))
    cond_A.append(np.linalg.cond(mc.analytic_jacobian(puma, qq)))
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(q5s, np.rad2deg(pitch)); ax[0].set_xlabel("$q_5$ [rad]"); ax[0].set_ylabel("pitch [deg]")
ax[1].semilogy(q5s, cond_G, label="geometric $J$"); ax[1].semilogy(q5s, cond_A, label="analytic $J_A$")
ax[1].set_xlabel("$q_5$ [rad]"); ax[1].set_ylabel("condition number"); ax[1].legend()
plt.tight_layout(); plt.show()

## What to remember

- Column $i$ of $J$ is the end-effector velocity due to joint $i$ alone: $z_{i-1}\times(p - o_{i-1})$ and $z_{i-1}$.
- For a link Jacobian use that link's origin, and zero the columns of joints beyond it.
- $J_O$ is angular velocity, not a derivative of angles; the analytic Jacobian adds $T^{-1}$ and inherits the
  gimbal-lock singularity.
- Check every new Jacobian with central differences, $h \approx 10^{-6}$.